# DRL Heating Process — Open Reproducibility Companion

This notebook demonstrates a **synthetic conceptual reproduction** of the workflow described in El Mazgualdi et al., *A Deep Reinforcement Learning (DRL) Decision Model for Heating Process Parameters Identification in Automotive Glass Manufacturing*, Springer, DOI: **10.1007/978-3-030-51186-9_6**.

**Important:** this is not the original industrial dataset or author-supplied code. It reproduces the high-level architecture (ANN process surrogate + DQN recipe controller) on a transparent synthetic process.

In [ ]:
from pathlib import Path
import sys, subprocess

if Path('drl_heating_repro.py').exists():
    module_dir = Path.cwd()
elif Path('reproducibility/drl_heating/drl_heating_repro.py').exists():
    module_dir = Path('reproducibility/drl_heating').resolve()
else:
    repo = Path('/content/AI-research')
    if not repo.exists():
        subprocess.run(['git', 'clone', '-q', 'https://github.com/MasrourTawfik/AI.git', str(repo)], check=True)
    module_dir = repo / 'reproducibility' / 'drl_heating'
sys.path.insert(0, str(module_dir))
print('Using companion code from:', module_dir)

In [ ]:
import numpy as np
import torch
from drl_heating_repro import Config, HeatingEnv, evaluate, set_seed, train_dqn, train_surrogate
cfg = Config(dqn_episodes=140)
set_seed(cfg.seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. Train the process-surrogate ANN

The surrogate learns the included transparent synthetic furnace generator. This mirrors the paper's use of a self-prediction ANN without claiming to reproduce proprietary furnace data.

In [ ]:
surrogate, surrogate_metrics = train_surrogate(cfg, device)
surrogate_metrics

## 2. Train the DQN recipe controller

The controller adjusts **glass transfer speed** and **zone temperature** to reach a requested outlet temperature.

In [ ]:
env = HeatingEnv(surrogate, cfg, device)
q, history = train_dqn(env, cfg, device)
last50 = np.array(history['final_abs_error_C'][-50:])
{'last_50_mean_abs_error_C': float(last50.mean()), 'last_50_success_rate': float((last50 <= cfg.tolerance).mean())}

## 3. Evaluate target temperatures

In [ ]:
targets = [440., 460., 480., 500., 520., 535.]
results = evaluate(env, q, targets, device)
results

In [ ]:
import pandas as pd
pd.DataFrame(results)

## Citation

Please cite the **original Springer chapter**, not this notebook:

El Mazgualdi, C., Masrour, T., El Hassani, I., & Khdoudi, A. (2021). *A Deep Reinforcement Learning (DRL) Decision Model for Heating Process Parameters Identification in Automotive Glass Manufacturing*. In **Artificial Intelligence and Industrial Applications: Smart Operation Management**, pp. 77–87. Springer. https://doi.org/10.1007/978-3-030-51186-9_6